Cell 1 – Install / Import

In [3]:
# ============================================================
# 🔥 Telco Churn Prediction — PyTorch
# Diamond • Ultimate • Elite • Netflix‑Ready
# ============================================================

# === Cell 1: Imports ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from datetime import datetime

# === Cell 2: Config ===
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = "../backend/models/telco/"
MODEL_NAME = "torch_model.pt"
FEATURES_NAME = "feature_names_torch.json"

os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🖥️ Using device:", device)

# === Cell 3: Load Data ===
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded dataset: {df.shape}")

# === Cell 4: Clean & Preprocess ===
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df = df.drop(columns=["customerID"])

y = df["Churn"]
X = df.drop(columns=["Churn"])

# One‑hot encode categoricals (MATCH sklearn/xgb)
X = pd.get_dummies(X)

# Save feature names
feature_names = X.columns.tolist()

# Scale numerics
scaler = StandardScaler()
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
X[num_cols] = scaler.fit_transform(X[num_cols])

# === 🛠️ Extra Fix: Convert to numeric explicitly ===
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

# === Cell 5: Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Convert to tensors ===
X_train = torch.tensor(X_train.values.astype(np.float32))
X_test = torch.tensor(X_test.values.astype(np.float32))
y_train = torch.tensor(y_train.values.astype(np.int64))
y_test = torch.tensor(y_test.values.astype(np.int64))

# === Cell 6: DataLoaders ===
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

# === Cell 7: Model ===
class TelcoChurnNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

model = TelcoChurnNet(input_dim=X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# === Cell 8: Training ===
epochs = 25
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}")

# === Cell 9: Evaluation ===
model.eval()
preds_all, labels_all = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        preds_all.extend(preds)
        labels_all.extend(yb.numpy())

print("📊 Classification Report:")
print(classification_report(labels_all, preds_all, digits=4))
print("🧮 Confusion Matrix:")
print(confusion_matrix(labels_all, preds_all))

# === Cell 10: Save Model & Metadata ===
torch.save(model.state_dict(), os.path.join(MODEL_DIR, MODEL_NAME))

with open(os.path.join(MODEL_DIR, FEATURES_NAME), "w") as f:
    json.dump(feature_names, f, indent=2)

print("✅ PyTorch model saved:", MODEL_NAME)
print("🧠 Feature names saved:", FEATURES_NAME)
print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

🖥️ Using device: cpu
✅ Loaded dataset: (7043, 21)
Epoch 1/25 — Loss: 0.4772
Epoch 2/25 — Loss: 0.4234
Epoch 3/25 — Loss: 0.4199
Epoch 4/25 — Loss: 0.4144
Epoch 5/25 — Loss: 0.4135
Epoch 6/25 — Loss: 0.4129
Epoch 7/25 — Loss: 0.4096
Epoch 8/25 — Loss: 0.4066
Epoch 9/25 — Loss: 0.4060
Epoch 10/25 — Loss: 0.4042
Epoch 11/25 — Loss: 0.4033
Epoch 12/25 — Loss: 0.4009
Epoch 13/25 — Loss: 0.3988
Epoch 14/25 — Loss: 0.3979
Epoch 15/25 — Loss: 0.3952
Epoch 16/25 — Loss: 0.3944
Epoch 17/25 — Loss: 0.3914
Epoch 18/25 — Loss: 0.3903
Epoch 19/25 — Loss: 0.3873
Epoch 20/25 — Loss: 0.3821
Epoch 21/25 — Loss: 0.3872
Epoch 22/25 — Loss: 0.3813
Epoch 23/25 — Loss: 0.3840
Epoch 24/25 — Loss: 0.3787
Epoch 25/25 — Loss: 0.3771
📊 Classification Report:
              precision    recall  f1-score   support

           0     0.8364    0.8906    0.8626      1033
           1     0.6319    0.5187    0.5698       374

    accuracy                         0.7918      1407
   macro avg     0.7341    0.7047    0.71